# 04. 가계금융복지조사 마이크로데이터 분석 준비

이 노트북은 가계금융복지조사 마이크로데이터를 활용해 같은 가구 안에서 소득과 순자산의 관계를 확인하기 위한 준비 노트북입니다.

현재 프로젝트의 중심 데이터는 KGSS입니다. 가계금융복지조사 마이크로데이터는 KGSS에서 나타난 성공 인식 변화를 이해하기 위한 경제적 배경 자료로 사용합니다.

주의: 이 노트북은 원자료나 개인/가구 단위 processed data를 저장하지 않습니다. 저장 가능한 것은 집계표와 그림뿐입니다.


## 1. 라이브러리 불러오기

2025년 MDIS 가계금융복지조사 가구마스터 CSV를 분석합니다. 원자료는 로컬 `data/raw/`에만 두고, 저장 산출물은 집계표와 그림으로 제한합니다.

In [ ]:
from pathlib import Path
import os
import zipfile

# 샌드박스/로컬 실행 환경에서 matplotlib 캐시 경고를 줄입니다.
os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'matplotlib'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

try:
    import pyreadstat
except ImportError:
    pyreadstat = None

## 2. 경로 설정

MDIS에서 내려받은 원자료는 Git에 올리지 않습니다. 가능하면 `data/raw/household_finance_2025/` 아래에 로컬로만 둡니다.

아직 파일을 받지 않았다면 이 노트북은 분석을 실행하지 않고, 필요한 파일 안내만 출력합니다.


In [ ]:
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
raw_dir = project_root / 'data' / 'raw' / 'household_finance_2025'
downloads_dir = Path.home() / 'Downloads'
outputs_tables = project_root / 'outputs' / 'tables'
outputs_figures = project_root / 'outputs' / 'figures'
outputs_tables.mkdir(parents=True, exist_ok=True)
outputs_figures.mkdir(parents=True, exist_ok=True)

font_candidates = ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR', 'Noto Sans KR']
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
selected_font = next((font for font in font_candidates if font in installed_fonts), 'DejaVu Sans')
plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False

print('프로젝트 루트:', project_root)
print('원자료 권장 위치:', raw_dir)
print('선택된 한글 폰트:', selected_font)

## 3. 로컬 마이크로데이터 파일 찾기

아래 셀은 `data/raw/household_finance_2025/`와 다운로드 폴더에서 가계금융복지조사 관련 파일을 찾습니다.

원자료 파일은 Git에 보이면 안 됩니다. `data/raw/`는 `.gitignore`에 포함되어 있으므로, 원자료는 로컬에만 남아야 합니다.


In [ ]:
patterns = [
    '*가계금융*',
    '*금융복지*',
    '*가구마스터*',
    '*파일설계서*',
    '*household*finance*',
    '*Household*Finance*',
    '*MDIS*',
]
extensions = {'.csv', '.txt', '.xlsx', '.xls', '.sav', '.dta', '.sas7bdat', '.zip'}

candidate_files = []
for base in [raw_dir, downloads_dir]:
    if base.exists():
        for pattern in patterns:
            for path in base.rglob(pattern):
                if path.is_file() and path.suffix.lower() in extensions:
                    candidate_files.append(path)

candidate_files = sorted(set(candidate_files))
for i, path in enumerate(candidate_files, start=1):
    print(f'{i:02d}. {path}')

DATA_AVAILABLE = len(candidate_files) > 0
if not DATA_AVAILABLE:
    print()
    print('MDIS 마이크로데이터 파일을 찾지 못했습니다.')
    print('MDIS에서 가계금융복지조사 2025년 가구마스터 CSV와 파일설계서/코드집을 내려받은 뒤 다시 실행하세요.')

## 4. 압축 파일 내용 확인

MDIS 자료가 ZIP으로 제공될 수 있으므로, ZIP 파일이 있으면 내부 파일명을 먼저 확인합니다. 이 단계에서는 압축을 풀지 않고 목록만 확인합니다.


In [ ]:
zip_files = [path for path in candidate_files if path.suffix.lower() == '.zip']
for zip_path in zip_files:
    print(f'\nZIP: {zip_path}')
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist()[:50]:
            print(' -', name)
        if len(zf.namelist()) > 50:
            print(' ...')


## 5. 분석용 데이터 파일 선택

2025년 가계금융복지조사 가구마스터 CSV와 코드집 포함 파일설계서를 사용합니다.

- CSV 인코딩: `cp949`
- 원자료 위치: `data/raw/household_finance_2025/`
- 분석 산출물: `outputs/tables/`, `outputs/figures/`

In [ ]:
microdata_path = raw_dir / '2025_가구마스터_20260604_10391.csv'
codebook_path = raw_dir / '파일설계서(공공용)_가계금융복지조사(2017년~)_가구마스터(제공)_2025(코드집포함).xlsx'

print('microdata_path:', microdata_path)
print('microdata exists:', microdata_path.exists())
print('codebook_path:', codebook_path)
print('codebook exists:', codebook_path.exists())

## 6. 파일 구조 미리보기

파일 형식에 따라 처음 몇 행 또는 메타데이터만 읽습니다. 이 단계는 변수명을 확인하기 위한 단계이며, 개인/가구 단위 자료를 저장하지 않습니다.


In [ ]:
def read_microdata_preview(path, nrows=5):
    suffix = path.suffix.lower()
    if suffix in ['.csv', '.txt']:
        try:
            return pd.read_csv(path, nrows=nrows, encoding='utf-8-sig')
        except UnicodeDecodeError:
            return pd.read_csv(path, nrows=nrows, encoding='cp949')
    if suffix in ['.xlsx', '.xls']:
        return pd.read_excel(path, nrows=nrows)
    if suffix == '.sav':
        if pyreadstat is None:
            raise ImportError('pyreadstat이 설치되어 있지 않아 SAV 파일을 읽을 수 없습니다.')
        df, meta = pyreadstat.read_sav(path, row_limit=nrows)
        return df, meta
    if suffix == '.dta':
        return pd.read_stata(path, iterator=True).read(nrows=nrows)
    if suffix == '.sas7bdat':
        if pyreadstat is None:
            raise ImportError('pyreadstat이 설치되어 있지 않아 SAS 파일을 읽을 수 없습니다.')
        df, meta = pyreadstat.read_sas7bdat(path, row_limit=nrows)
        return df, meta
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')

if microdata_path is not None and microdata_path.exists():
    preview = read_microdata_preview(microdata_path)
    if isinstance(preview, tuple):
        preview_df, preview_meta = preview
        display(preview_df.head())
        print('columns:', list(preview_df.columns)[:50])
    else:
        preview_df = preview
        preview_meta = None
        display(preview_df.head())
        print('columns:', list(preview_df.columns)[:50])
else:
    preview_df = None
    preview_meta = None
    print('파일이 지정되지 않아 미리보기를 건너뜁니다.')

## 7. 변수 후보 자동 검색

파일을 읽은 뒤, 변수명과 라벨에서 필요한 변수 후보를 찾습니다. 정확한 변수명은 파일설계서와 코드집을 보고 최종 확정해야 합니다.


In [ ]:
def search_columns(columns, keywords):
    results = []
    for col in columns:
        text = str(col).lower()
        if any(keyword.lower() in text for keyword in keywords):
            results.append(col)
    return results

if preview_df is not None:
    columns = list(preview_df.columns)
    keyword_groups = {
        'age': ['age', '연령', '가구주', 'h_age'],
        'household_income': ['income', '소득', '가구소득', '총소득'],
        'labor_income': ['labor', 'earned', '근로', '근로소득'],
        'asset': ['asset', '자산'],
        'net_asset': ['net', '순자산'],
        'debt': ['debt', '부채'],
        'weight': ['weight', 'wt', '가중'],
        'housing': ['house', 'home', '주택', '자가', '점유'],
    }
    for group, keywords in keyword_groups.items():
        print(f'\n[{group}]')
        print(search_columns(columns, keywords))
else:
    print('preview_df가 없어 변수 후보 검색을 건너뜁니다.')


## 8. 변수명 매핑

파일설계서와 CSV 컬럼명을 확인한 결과, 이번 분석에 필요한 변수는 모두 2025년 가구마스터에 포함되어 있습니다.

In [ ]:
variable_map = {
    'year': '조사연도',
    'household_id': 'MD제공용_가구고유번호',
    'weight': '가중값',
    'age': '가구주_만연령',
    'age_group': '가구주연령_10세단위코드',
    'sex': '가구주_성별코드',
    'education': '가구주_교육정도_통합코드',
    'employment_status': '가구주_종사상지위코드',
    'marital_status': '가구주_혼인상태코드',
    'tenure': '입주형태통합코드',
    'housing_type': '주택종류통합코드',
    'income_quintile': '소득5분위코드(보완)',
    'income_decile': '소득10분위코드(보완)(2017년~)',
    'net_asset_quintile': '순자산5분위코드',
    'net_asset_decile': '순자산10분위코드',
    'asset': '자산',
    'debt': '부채',
    'net_asset': '순자산',
    'current_income': '경상소득(보완)',
    'labor_income': '경상소득_근로소득(보완)',
    'business_income': '경상소득_사업소득(보완)',
    'property_income': '경상소득_재산소득(보완)',
    'financial_asset': '자산_금융자산',
    'real_asset': '자산_실물자산',
    'real_estate': '자산_실물자산_부동산금액',
    'primary_home_value': '자산_실물자산_부동산_거주주택금액',
    'financial_debt': '부채_금융부채',
    'secured_loan': '부채_금융부채_담보대출금액',
}

if preview_df is not None:
    missing_columns = [col for col in variable_map.values() if col not in preview_df.columns]
    if missing_columns:
        raise KeyError(f'필수 변수가 없습니다: {missing_columns}')
    print(f'필수 변수 {len(variable_map)}개가 모두 확인되었습니다.')
else:
    print('preview_df가 없어 변수 확인을 건너뜁니다.')

pd.Series(variable_map, name='csv_column')

## 9. 코드 라벨 불러오기

코드값(`Q1`, `G1` 등)을 사람이 읽을 수 있는 라벨로 바꾸기 위해 파일설계서의 `코드정보` 시트를 사용합니다.

In [ ]:
def load_codebook(path):
    if not path.exists():
        print('코드북 파일이 없어 코드 라벨 매핑을 건너뜁니다.')
        return pd.DataFrame(columns=['코드번호', '항목명', '코드', '코드의미 및 단위', '특이사항'])
    code_df = pd.read_excel(path, sheet_name='코드정보', header=1)
    code_df = code_df.dropna(subset=['항목명', '코드']).copy()
    code_df['코드'] = code_df['코드'].astype(str)
    code_df['코드의미 및 단위'] = code_df['코드의미 및 단위'].astype(str)
    return code_df

codebook = load_codebook(codebook_path)

def code_map(item_name):
    sub = codebook[codebook['항목명'].eq(item_name)]
    return dict(zip(sub['코드'].astype(str), sub['코드의미 및 단위'].astype(str)))

label_maps = {
    'age_group': code_map(variable_map['age_group']),
    'tenure': code_map(variable_map['tenure']),
    'housing_type': code_map(variable_map['housing_type']),
    'income_quintile': code_map(variable_map['income_quintile']),
    'net_asset_quintile': code_map(variable_map['net_asset_quintile']),
}

for key, mapping in label_maps.items():
    print(f'[{key}]', mapping)

## 10. 원자료 읽기 및 분석 변수 생성

분석은 2025년 단면 기준입니다. 분위 변수는 MDIS 제공 분위 코드를 사용하고, 가중값을 적용해 비율을 계산합니다.

In [ ]:
def read_household_microdata(path):
    if not path.exists():
        raise FileNotFoundError(f'마이크로데이터 파일이 없습니다: {path}')
    return pd.read_csv(path, encoding='cp949')

hf = read_household_microdata(microdata_path)

required_columns = list(variable_map.values())
missing_columns = [col for col in required_columns if col not in hf.columns]
if missing_columns:
    raise KeyError(f'필수 변수가 없습니다: {missing_columns}')

analysis = hf[required_columns].copy()
analysis['income_top40'] = analysis[variable_map['income_quintile']].isin(['Q4', 'Q5'])
analysis['income_low40'] = analysis[variable_map['income_quintile']].isin(['Q1', 'Q2'])
analysis['net_asset_top40'] = analysis[variable_map['net_asset_quintile']].isin(['Q4', 'Q5'])
analysis['net_asset_low40'] = analysis[variable_map['net_asset_quintile']].isin(['Q1', 'Q2'])
analysis['income_top40_net_asset_low40'] = analysis['income_top40'] & analysis['net_asset_low40']
analysis['income_low40_net_asset_top40'] = analysis['income_low40'] & analysis['net_asset_top40']
analysis['both_top40'] = analysis['income_top40'] & analysis['net_asset_top40']
analysis['both_low40'] = analysis['income_low40'] & analysis['net_asset_low40']

print('자료 크기:', hf.shape)
print('분석 변수 크기:', analysis.shape)
print('조사연도:', sorted(analysis[variable_map['year']].unique()))

## 11. 가중 집계 함수

개별 가구 행은 저장하지 않고, 가중 비율이 계산된 집계표만 저장합니다.

In [ ]:
def weighted_sum(mask, weights):
    return weights[mask].sum()

def weighted_pct(mask, weights, base_mask=None):
    if base_mask is None:
        base_mask = pd.Series(True, index=weights.index)
    denominator = weights[base_mask].sum()
    if denominator == 0:
        return np.nan
    return weights[mask & base_mask].sum() / denominator * 100

def mismatch_summary(data, group_col=None, label_map=None):
    weights = data[variable_map['weight']]
    metrics = {
        '소득상위40_순자산하위40': data['income_top40_net_asset_low40'],
        '소득하위40_순자산상위40': data['income_low40_net_asset_top40'],
        '소득상위40_순자산상위40': data['both_top40'],
        '소득하위40_순자산하위40': data['both_low40'],
    }

    if group_col is None:
        rows = []
        for metric, mask in metrics.items():
            rows.append({
                '구분': '전체',
                '지표': metric,
                '가중비율_pct': weighted_pct(mask, weights),
                '비가중_가구수': int(mask.sum()),
            })
        return pd.DataFrame(rows)

    rows = []
    for group_value, group_index in data.groupby(group_col, dropna=False).groups.items():
        base_mask = data.index.isin(group_index)
        group_label = label_map.get(str(group_value), str(group_value)) if label_map else str(group_value)
        for metric, mask in metrics.items():
            rows.append({
                '구분코드': group_value,
                '구분': group_label,
                '지표': metric,
                '가중비율_pct': weighted_pct(mask, weights, base_mask),
                '비가중_가구수': int((mask & base_mask).sum()),
                '전체_비가중_가구수': int(base_mask.sum()),
            })
    return pd.DataFrame(rows)

def weighted_crosstab_pct(data, row_col, col_col):
    table = pd.crosstab(
        data[row_col],
        data[col_col],
        values=data[variable_map['weight']],
        aggfunc='sum',
        normalize='all',
    ) * 100
    return table.reindex(index=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'], columns=['Q1', 'Q2', 'Q3', 'Q4', 'Q5']).round(2)

print('집계 함수 준비 완료')

## 12. 소득 분위 × 순자산 분위 교차표

소득 분위와 순자산 분위가 같은 방향으로만 움직이지 않는지 확인합니다.

In [ ]:
income_net_asset_crosstab = weighted_crosstab_pct(
    analysis,
    variable_map['income_quintile'],
    variable_map['net_asset_quintile'],
)
income_net_asset_crosstab.index = [label_maps['income_quintile'].get(idx, idx) for idx in income_net_asset_crosstab.index]
income_net_asset_crosstab.columns = [label_maps['net_asset_quintile'].get(col, col) for col in income_net_asset_crosstab.columns]

crosstab_path = outputs_tables / '04_income_net_asset_quintile_crosstab_2025.csv'
income_net_asset_crosstab.to_csv(crosstab_path, encoding='utf-8-sig')

print('저장:', crosstab_path)
display(income_net_asset_crosstab)

## 13. 소득-순자산 불일치 집단 비율

핵심 관심 집단은 다음 두 집단입니다.

- 소득 상위 40% + 순자산 하위 40%
- 소득 하위 40% + 순자산 상위 40%

In [ ]:
overall_mismatch = mismatch_summary(analysis)
age_mismatch = mismatch_summary(
    analysis,
    variable_map['age_group'],
    label_maps['age_group'],
)
tenure_mismatch = mismatch_summary(
    analysis,
    variable_map['tenure'],
    label_maps['tenure'],
)

summary_path = outputs_tables / '04_income_net_asset_mismatch_summary_2025.csv'
age_path = outputs_tables / '04_income_net_asset_mismatch_by_age_2025.csv'
tenure_path = outputs_tables / '04_income_net_asset_mismatch_by_tenure_2025.csv'

overall_mismatch.to_csv(summary_path, index=False, encoding='utf-8-sig')
age_mismatch.to_csv(age_path, index=False, encoding='utf-8-sig')
tenure_mismatch.to_csv(tenure_path, index=False, encoding='utf-8-sig')

print('저장:', summary_path)
print('저장:', age_path)
print('저장:', tenure_path)

display(overall_mismatch.round({'가중비율_pct': 2}))
display(age_mismatch.round({'가중비율_pct': 2}).head(12))
display(tenure_mismatch.round({'가중비율_pct': 2}).head(12))

## 14. PPT 후보 그림 저장

PPT에서는 경제 분석을 보조 근거로만 사용하므로, 핵심 그림은 1~2개만 선별하는 것이 좋습니다.

In [ ]:
def save_heatmap(table, path):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    im = ax.imshow(table.values, cmap='Blues')
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=35, ha='right')
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.set_xlabel('순자산 5분위')
    ax.set_ylabel('소득 5분위')
    ax.set_title('소득 5분위 × 순자산 5분위 교차 비율(2025)')
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, f'{table.iloc[i, j]:.1f}%', ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='가중 비율(%)')
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()


def save_mismatch_bar(summary, path, title):
    focus = summary[summary['지표'].isin([
        '소득상위40_순자산하위40',
        '소득하위40_순자산상위40',
    ])].copy()
    pivot = focus.pivot(index='구분', columns='지표', values='가중비율_pct')
    pivot = pivot[[
        '소득상위40_순자산하위40',
        '소득하위40_순자산상위40',
    ]]
    if '구분코드' in focus.columns:
        label_order = focus[['구분코드', '구분']].drop_duplicates().sort_values('구분코드')['구분'].tolist()
        pivot = pivot.reindex(label_order)
    ax = pivot.plot(kind='bar', figsize=(8, 4.8), width=0.75)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('가중 비율(%)')
    ax.legend(['소득 상위40%·순자산 하위40%', '소득 하위40%·순자산 상위40%'], frameon=False)
    ax.tick_params(axis='x', rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f', padding=3, fontsize=9)
    ax.figure.tight_layout()
    ax.figure.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()

heatmap_path = outputs_figures / '04_income_net_asset_quintile_heatmap_2025.png'
age_fig_path = outputs_figures / '04_income_net_asset_mismatch_by_age_2025.png'
tenure_fig_path = outputs_figures / '04_income_net_asset_mismatch_by_tenure_2025.png'

save_heatmap(income_net_asset_crosstab, heatmap_path)
save_mismatch_bar(age_mismatch, age_fig_path, '연령대별 소득-순자산 불일치 집단 비율(2025)')
save_mismatch_bar(tenure_mismatch, tenure_fig_path, '입주형태별 소득-순자산 불일치 집단 비율(2025)')

print('저장:', heatmap_path)
print('저장:', age_fig_path)
print('저장:', tenure_fig_path)

## 15. 해석 메모

- 이 결과는 자산 격차가 성공 인식 변화를 일으켰다는 인과 증거가 아닙니다.
- 소득과 순자산의 불일치는 연령, 생애주기, 주택 보유 및 입주 형태와 함께 해석해야 합니다.
- KGSS가 프로젝트의 중심 자료이며, 이 분석은 성공 인식 변화가 등장하는 경제적 배경을 보완합니다.

In [ ]:
interpretation_points = [
    '2025년 가구 단면에서 소득 상위 40%이면서 순자산 하위 40%인 가구가 존재한다.',
    '소득 하위 40%이면서 순자산 상위 40%인 가구도 존재하며, 이 비율은 60세 이상에서 높다.',
    '따라서 순자산은 현재 소득만이 아니라 연령·주택·생애주기 효과가 섞인 지표로 해석해야 한다.',
    '이 분석은 KGSS 성공 인식 분석의 보조 자료이며 인과 주장에는 사용하지 않는다.',
]

for point in interpretation_points:
    print('-', point)